In [1]:
import pandas as pd
import nltk 
from nltk.tokenize import word_tokenize,sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder,OneHotEncoder
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv(r'ml_resume_dataset_4500.csv')

In [3]:
df.sample(5)

,id,name,years_experience,highest_degree,skills,current_title,has_portfolio,raw_text,label
2734,2735,Hasan Molla,3.0,Masters,"JSON, Kubernetes, LightGBM, Machine Learning, ...",Applied ML Engineer,True,Experienced ML engineer with strong background...,1
3030,3031,Sadia Das,0.0,Bachelors,"Linux, FastAPI, GPT, XGBoost, REST APIs, Sciki...",Receptionist,False,Customer support representative with some Exce...,0
1235,1236,Firoz Raihan,1.0,Bachelors,"Speech Recognition, OpenCV, Clustering, Signal...",Office Assistant,False,Customer support representative with some Exce...,0
501,502,Kawsar Islam,9.0,Masters,"REST APIs, SQL, BERT, EDA, Docker Compose, Clu...",Receptionist,False,Marketing intern with experience in social med...,0
1433,1434,Jony Kabir,5.0,Masters,"PyTorch, JSON",Administrative Assistant,False,Marketing intern with experience in social med...,0


In [4]:
df= df.drop(columns = ['id','name'])

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   years_experience  4500 non-null   float64
 1   highest_degree    4500 non-null   str    
 2   skills            4500 non-null   str    
 3   current_title     4500 non-null   str    
 4   has_portfolio     4500 non-null   bool   
 5   raw_text          4500 non-null   str    
 6   label             4500 non-null   int64  
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 215.5 KB


In [6]:
import numpy as np
label = LabelEncoder()
Ord = OrdinalEncoder()
a= np.array(df['highest_degree'])
a = a.reshape(-1,1)
df['has_portfolio_encoded'] = label.fit_transform(df['has_portfolio'])
df['highest_degree_encoded'] = Ord.fit_transform(a)
df = df.drop(columns = ['has_portfolio','highest_degree'])



In [7]:
df.sample()

,years_experience,skills,current_title,raw_text,label,has_portfolio_encoded,highest_degree_encoded
2377,2.0,"EDA, Time Series, Reinforcement Learning, Data...",Data Entry Operator,Junior intern with exposure to data labeling a...,0,0,0.0


## Making Functions for NLP

In [8]:
def rem_stopwords(text):
    if pd.isna(text):
        return ''
    words  = stopwords.words('english')
    return " ".join(
        word for word in text.split()
        if word.lower() not in words
         
    )

    
def word_tokens(text):
    return " ".join(word_tokenize(text))



In [9]:
df['raw_text_new'] = df['raw_text'].apply(rem_stopwords)
df= df.drop(columns = ['raw_text'])

In [10]:
df['skills'] = df['skills'].apply(word_tokens)
df['raw_text_new'] = df['raw_text_new'].apply(word_tokens)

In [11]:
X = df.drop(columns= ['label'])
y = df['label']

In [12]:
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import OneHotEncoder

# 1. Combine your text fields into ONE text column
X['text_combined'] = X['skills'] + ' ' + X['raw_text_new']
X = X.drop(columns=['skills', 'raw_text_new'])

# 2. current_title is also text/categorical — needs its own encoding (it's NOT numeric)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# 3. Split BEFORE fitting any vectorizer/encoder (you already do this — good practice)
X_tr, X_tst, y_tr, y_tst = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Fit TF-IDF on the text column ONLY (a Series, not a DataFrame)
tfidf = TfidfVectorizer(max_features=5000)
text_tr = tfidf.fit_transform(X_tr['text_combined'])
text_tst = tfidf.transform(X_tst['text_combined'])

# 5. Encode current_title separately
title_tr = ohe.fit_transform(X_tr[['current_title']])
title_tst = ohe.transform(X_tst[['current_title']])

# 6. Numeric columns as a sparse matrix
num_cols = ['years_experience', 'has_portfolio_encoded', 'highest_degree_encoded']
num_tr = csr_matrix(X_tr[num_cols].values)
num_tst = csr_matrix(X_tst[num_cols].values)

# 7. Stack everything together horizontally
X_tr_final = hstack([text_tr, title_tr, num_tr])
X_tst_final = hstack([text_tst, title_tst, num_tst])

# 8. Now fit
lg = LogisticRegression(max_iter=1000)
lg.fit(X_tr_final, y_tr)
print(lg.score(X_tst_final, y_tst))


1.0


In [15]:
y_pred = lg.predict(X_tst_final)
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [16]:
accuracy_score(y_tst,y_pred)

1.0

In [17]:
print(classification_report(y_tst,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       642
           1       1.00      1.00      1.00       258

    accuracy                           1.00       900
   macro avg       1.00      1.00      1.00       900
weighted avg       1.00      1.00      1.00       900



In [18]:
# fit a model using ONLY current_title, nothing else
ohe_only = OneHotEncoder(handle_unknown='ignore')
title_only_tr = ohe_only.fit_transform(X_tr[['current_title']])
title_only_tst = ohe_only.transform(X_tst[['current_title']])

lg2 = LogisticRegression(max_iter=1000)
lg2.fit(title_only_tr, y_tr)
print(lg2.score(title_only_tst, y_tst))

1.0


In [19]:
# Rebuild features WITHOUT current_title
X_tr_no_title = hstack([text_tr, num_tr])      # drop title_tr
X_tst_no_title = hstack([text_tst, num_tst])   # drop title_tst

lg3 = LogisticRegression(max_iter=1000)
lg3.fit(X_tr_no_title, y_tr)

y_pred3 = lg3.predict(X_tst_no_title)
print(accuracy_score(y_tst, y_pred3))
print(classification_report(y_tst, y_pred3))

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       642
           1       1.00      1.00      1.00       258

    accuracy                           1.00       900
   macro avg       1.00      1.00      1.00       900
weighted avg       1.00      1.00      1.00       900



In [20]:
df.groupby('raw_text_new')['label'].nunique().value_counts()

label
1    20
Name: count, dtype: int64